In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.historical_analysis.dataScraper import *


pd.set_option('display.max_columns', None)

In [2]:
today = datetime.today().strftime('%Y%m%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

us_file = get_latest_file('NBA_US_*.csv')
dfs_file = get_latest_file('NBA_DFS_*.csv')

if us_file is None:
    raise ValueError("No US file found")

if dfs_file is None:
    raise ValueError("No DFS file found")

us_df = pd.read_csv(us_file)
lines_dfs = pd.read_csv(dfs_file)

lines_us = us_df[us_df['CATEGORY'] == 'player_points'].copy()


print("US file:", us_file.name)
print("DFS file:", dfs_file.name)
print("DFS latest pull:", lines_dfs['DATA_PULLED_AT'].max())
print("US latest pull:", us_df['DATA_PULLED_AT'].max())

US file: NBA_US_20260330_154051.csv
DFS file: NBA_DFS_20260330_154001.csv
DFS latest pull: 2026-03-30 15:40:01
US latest pull: 2026-03-30 15:40:51


In [3]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')

if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260330_154050.json


,home_team,away_team,commence_time,bookmakers
0,Miami Heat,Philadelphia 76ers,2026-03-30 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Atlanta Hawks,Boston Celtics,2026-03-30 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,San Antonio Spurs,Chicago Bulls,2026-03-31 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Memphis Grizzlies,Phoenix Suns,2026-03-31 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Dallas Mavericks,Minnesota Timberwolves,2026-03-31 00:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [4]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
132,NaN,2025-26,1630536,Sharife Cooper,Sharife,1610612764,WAS,Washington Wizards,22501087,2026-03-29T00:00:00,WAS @ POR,L,19.000000,4,9,0.444,0,1,0.000,2,2,1.000,0,1,1,2,2,0,0,0,1,2,10,-11,12.2,0,0,13.0,1,19:00,1,90.4,90.0,90.0,121.8,120.5,120.5,-31.4,-30.5,-30.5,0.250,1.0,14.3,0.000,0.056,0.027,14.3,14.4,0.444,0.506,0.279,0.291,99.08,99.79,83.16,99.79,0.077,40,4.0,9.0,32,87,0.368,5,22,0.227,19,22,0.864,9,30,39,16,16.0,7,2,10,28,24,88,-35.0,84.9,85.4,119.3,119.4,-34.4,-34.0,0.500,1.00,12.1,0.232,0.696,0.441,0.155,0.397,0.455,103.4,103.0,85.83,103,0.280,1610612757,POR,Portland Trail Blazers,44,86,0.512,12,36,0.333,23,32,0.719,11,38,49,24,14.0,8,10,2,24,28,123,35.0,119.3,119.4,84.9,85.4,34.4,34.0,0.545,1.71,17.0,0.304,0.768,0.559,0.136,0.581,0.615,103.4,103.0,85.83,103,0.720,NaN,PG,24.0
131,NaN,2025-26,1630264,Anthony Gill,Anthony,1610612764,WAS,Washington Wizards,22501087,2026-03-29T00:00:00,WAS @ POR,L,30.600000,4,7,0.571,0,1,0.000,0,0,0.000,0,3,3,0,2,1,0,1,4,1,8,-34,12.6,0,0,13.0,1,30:36,1,77.0,76.5,76.5,130.2,128.4,128.4,-53.2,-51.9,-51.9,0.000,0.0,0.0,0.000,0.103,0.049,22.2,22.2,0.571,0.571,0.122,0.126,104.75,105.88,88.24,105.88,0.026,68,4.0,7.0,32,87,0.368,5,22,0.227,19,22,0.864,9,30,39,16,16.0,7,2,10,28,24,88,-35.0,84.9,85.4,119.3,119.4,-34.4,-34.0,0.500,1.00,12.1,0.232,0.696,0.441,0.155,0.397,0.455,103.4,103.0,85.83,103,0.280,1610612757,POR,Portland Trail Blazers,44,86,0.512,12,36,0.333,23,32,0.719,11,38,49,24,14.0,8,10,2,24,28,123,35.0,119.3,119.4,84.9,85.4,34.4,34.0,0.545,1.71,17.0,0.304,0.768,0.559,0.136,0.581,0.615,103.4,103.0,85.83,103,0.720,NaN,PF,33.0
130,NaN,2025-26,1629021,Moritz Wagner,Moritz,1610612753,ORL,Orlando Magic,22501086,2026-03-29T00:00:00,ORL @ TOR,L,12.000000,3,6,0.500,1,2,0.500,0,0,0.000,0,4,4,1,0,0,0,1,0,1,7,-7,13.3,0,0,13.0,1,12:00,1,73.1,70.4,70.4,100.5,100.0,100.0,-27.4,-29.6,-29.6,0.200,0.0,14.3,0.000,0.333,0.160,0.0,0.0,0.583,0.583,0.222,0.231,103.76,106.00,88.33,106.00,0.162,27,3.0,6.0,31,82,0.378,9,38,0.237,16,19,0.842,9,27,36,20,28.0,4,1,3,20,18,87,-52.0,79.6,83.7,129.1,132.4,-49.5,-48.7,0.645,0.71,14.4,0.283,0.705,0.474,0.269,0.433,0.481,108.5,104.5,87.08,104,0.195,1610612761,TOR,Toronto Raptors,54,94,0.574,13,29,0.448,18,22,0.818,7,37,44,41,11.0,20,3,1,18,20,139,52.0,129.1,132.4,79.6,83.7,49.5,48.7,0.759,3.73,25.8,0.295,0.717,0.526,0.105,0.644,0.670,108.5,104.5,87.08,105,0.805,NaN,C,28.0
141,NaN,2025-2

In [9]:
# Change line_bookmaker to e.g. 'PrizePicks' to use that DFS book's lines from lines_dfs
# Opp Allowed (pos) / Line vs Opp Allowed need SportsData.io: export SPORTSDATA_IO_KEY or pass sportsdata_api_key=...
final, tier1_all, final = generalized_best_bets(
    lines_dfs, base_df, us_df, team_dds,
    line_bookmaker='PrizePicks',
)

# Player `POSITION` on each row is set inside generalized_best_bets:
#   merged['POSITION'] = merged['Position']   # latest S26 row per PLAYER_ID → correct per player
# Do not use merged['POSITION'] = df['Position'].iloc[0] — that would assign every player the same slot.

print('Total bets across categories:', len(final))
print('Tier 1 bets:', len(tier1_all))

if not final.empty:
    display(tier1_all.head(20))

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/historical_analysis/dataScraper.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/historical_analysis/dataScraper.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/Users/alexgonzalez/Documents/NBA-Prop-Predictor

Total bets across categories: 527
Tier 1 bets: 139


/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/historical_analysis/dataScraper.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/historical_analysis/dataScraper.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/Users/alexgonzalez/Documents/NBA-Prop-Predictor

,PLAYER_NAME,POSITION,TEAM_NAME,OPPONENT,HOME_AWAY,TEAM_SPREAD,GAME_TOTAL,CATEGORY,LINE,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES,MATCHUP_EDGE,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,TOTAL_BOOST,IS_UNDERDOG,BET_FLAG,COMMENCE_TIME
0,Quentin Grimes,NaN,Philadelphia 76ers,Miami Heat,AWAY,-2.0,242.5,player_points,9.5,-109,-108,0.522,0.519,19.8,20.0,14.40,5.0,4.90,7.96,10.3,10.5,-1.294,0.902,0.098,72.95,-81.13,0.8,0.9,0.73,0.67,33.04,3.92,0.23,0.07,2.25,0,True,2026-03-30
1,VJ Edgecombe,NaN,Philadelphia 76ers,Miami Heat,AWAY,-2.0,242.5,player_points,13.5,-106,-111,0.515,0.526,20.4,19.5,19.00,1.0,5.50,9.70,6.9,6.0,-0.711,0.761,0.239,47.89,-54.57,0.8,0.7,0.73,0.58,33.40,5.38,0.23,0.05,2.25,0,True,2026-03-30
2,Tre Jones,NaN,Chicago Bulls,San Antonio Spurs,AWAY,18.5,245.5,player_points,13.0,-137,-137,0.578,0.578,17.4,18.5,20.00,1.0,7.00,4.70,4.4,5.5,-0.936,0.825,0.175,42.72,-69.73,1.0,0.9,0.80,0.32,29.02,2.94,0.21,0.03,2.55,1,True,2026-03-31
3,Daniel Gafford,NaN,Dallas Mavericks,Minnesota Timberwolves,HOME,8.0,235.5,player_points,10.5,-121,100,0.548,0.500,15.2,14.0,10.00,5.0,-0.50,6.14,4.7,3.5,-0.765,0.778,0.222,42.10,-55.60,0.8,0.7,0.47,0.46,24.35,3.32,0.19,0.03,1.55,1,True,2026-03-31
4,Joel Embiid,NaN,Philadelphia 76ers,Miami Heat,AWAY,-2.0,242.5,player_points,28.5,-105,100,0.512,0.500,31.5,31.0,18.50,2.0,-10.00,5.25,3.0,2.5,-0.571,0.716,0.284,39.79,-43.20,0.6,0.7,0.73,0.41,32.92,4.37,0.35,0.05,2.25,0,True,2026-03-30
5,Collin Sexton,NaN,Chicago Bulls,San Antonio Spurs,AWAY,18.5,245.5,player_points,16.0,-137,-137,0.578,0.578,21.2,22.0,19.60,5.0,3.60,6.60,5.2,6.0,-0.788,0.785,0.215,35.80,-62.81,0.6,0.7,0.47,0.51,26.21,6.60,0.25,0.04,2.55,1,True,2026-03-31
6,Nickeil Alexander-Walker,NaN,Atlanta Hawks,Boston Celtics,HOME,-2.5,225.0,player_points,18.0,-137,-137,0.578,0.578,23.3,21.0,13.40,5.0,-4.60,6.96,5.3,3.0,-0.761,0.777,0.223,34.42,-61.42,0.8,0.8,0.73,0.32,33.29,5.19,0.21,0.04,0.50,0,True,2026-03-30
7,Matas Buzelis,NaN,Chicago Bulls,San Antonio Spurs,AWAY,18.5,245.5,player_points,18.5,-109,-112,0.522,0.528,22.7,20.5,8.67,3.0,-9.83,8.34,4.2,2.0,-0.504,0.693,0.307,32.88,-41.89,0.4,0.6,0.67,0.22,34.82,4.40,0.24,0.04,2.55,1,True,2026-03-31
8,Klay Thompson,NaN,Dallas Mavericks,Minnesota Timberwolves,HOME,8.0,235.5,player_points,10.5,-105,-112,0.512,0.528,13.4,13.5,9.25,4.0,-1.25,6.79,2.9,3.0,-0.427,0.665,0.335,29.83,-36.59,0.6,0.6,0.53,0.61,21.43,5.71,0.23,0.06,1.55,1,True,2026-03-31
9,De'Aaron Fox,NaN,San Antonio Spurs,Chicago Bulls,HOME,-18.5,245.5,player_points,16.5,-105,100,0.512,0.500,18.5,17.5,23.50,2.0,7.00,5.10,2.0,1.0,-0.392,0.652,0.348,27.30,-30.40,0.2,0.6,0.53,0.68,29.72,5.45,0.24,0.04,2.55,0,True,2026-03-31


In [10]:
rename_map = {
    'PLAYER_NAME': 'Player',
    'POSITION': 'Position',
    'CATEGORY': 'Prop',
    'LINE': 'Line',
    'OPPONENT': 'Opponent',
    'TEAM_SPREAD': 'Spread',
    'GAME_TOTAL': 'Total',
    'OPP_DEF_RATING': 'Opp Def Rating',
    'OPP_POS_ALLOWED': 'Opp Allowed (pos)',
    'LINE_VS_OPP_ALLOWED': 'Line vs Opp Allowed',
    'OPP_RANK_DEF_RATING': 'Opp Def Rank',
    'OPP_PACE': 'Opp Pace',
    'OPP_PACE_RANK': 'Opp Pace Rank',
    'ODDS_OVER': 'Odds Over',
    'ODDS_UNDER': 'Odds Under',
    'IMP_PROB_OVER': 'Implied Over',
    'IMP_PROB_UNDER': 'Implied Under',
    'AVG_STAT_L10': 'Avg Stat L10',
    'MED_STAT_L10': 'Med Stat L10',
    'STD_STAT_L10': 'Std Stat L10',
    'EDGE': 'Edge',
    'MED_EDGE': 'Med Edge',
    'Z_SCORE': 'Z Score',
    'PROB_OVER': 'Prob Over',
    'PROB_UNDER': 'Prob Under',
    'EV_OVER': 'EV Over',
    'EV_UNDER': 'EV Under',
    'OVER_RATE_L5': 'OVER L5',
    'OVER_RATE_L10': 'OVER L10',
    'OVER_RATE_L15': 'OVER L15',
    'OVER_RATE_SEASON': 'ALL SEASON',
    'AVG_MIN_L10': 'Avg Min L10',
    'STD_MIN_L10': 'Std Min L10',
    'AVG_USG_L10': 'Avg USG% L10',
    'STD_USG_L10': 'Std USG% L10',
    'MIN_CONSISTENCY': 'Min Consistency',
    'IS_UNDERDOG': 'Underdog',
    'AVG_STAT_VS_MATCHUP': 'Avg Stat vs Matchup',
    'MATCHUP_GAMES': 'Matchup Games',
}

df = final.rename(columns=rename_map)
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['Prop'] = df['Prop'].map(prop_label_map).fillna(df['Prop'])

df = df[[
    'Player',
    'Position',
    'Prop',
    'Line',
    'Opponent',
    'Odds Over',
    'Odds Under',
    'Implied Over',
    'Implied Under',
    'EV Over',
    'EV Under',
    'Avg Stat L10',
    'Med Stat L10',
    'Std Stat L10',
    'Z Score',
    'Prob Over',
    'Prob Under',
    'OVER L5',
    'OVER L10',
    'OVER L15',
    'Avg Min L10',
    'Std Min L10',
    'Avg USG% L10',
    'Std USG% L10',
    'Avg Stat vs Matchup',
    'Matchup Games',
    'Spread',
    'Total',
    'Opp Def Rating',
    'Opp Def Rank',
    'Opp Pace',
    'Opp Pace Rank',
]].sort_values(by='EV Over', ascending=False)
df.head(10)

,Player,Position,Prop,Line,Opponent,Odds Over,Odds Under,Implied Over,Implied Under,EV Over,EV Under,Avg Stat L10,Med Stat L10,Std Stat L10,Z Score,Prob Over,Prob Under,OVER L5,OVER L10,OVER L15,Avg Min L10,Std Min L10,Avg USG% L10,Std USG% L10,Avg Stat vs Matchup,Matchup Games,Spread,Total,Opp Def Rating,Opp Def Rank,Opp Pace,Opp Pace Rank
0,Quentin Grimes,NaN,PTS,9.5,Miami Heat,-109,-108,0.522,0.519,72.95,-81.13,19.8,20.0,7.96,-1.294,0.902,0.098,0.8,0.9,0.73,33.04,3.92,0.23,0.07,14.4,5.0,-2.0,242.5,112.8,9,104.50,1
212,Tre Jones,NaN,PTS+REB+AST,21.5,San Antonio Spurs,102,-118,0.495,0.541,64.83,-66.01,27.3,27.0,6.43,-0.902,0.816,0.184,1.0,0.9,0.80,29.02,2.94,0.21,0.03,34.0,1.0,18.5,245.5,110.1,3,100.82,12
85,Jonathan Kuminga,NaN,REB,4.5,Boston Celtics,116,-122,0.463,0.550,63.94,-56.15,6.1,6.0,2.28,-0.702,0.759,0.241,0.6,0.8,0.73,21.59,3.82,0.21,0.03,3.0,2.0,-2.5,225.0,111.5,4,95.41,30
390,Quentin Grimes,NaN,PTS+AST,12.5,Miami Heat,-125,-102,0.556,0.505,62.54,-80.79,23.4,22.5,8.40,-1.298,0.903,0.097,0.8,0.9,0.73,33.04,3.92,0.23,0.07,17.4,5.0,-2.0,242.5,112.8,9,104.50,1
300,Tre Jones,NaN,PTS+REB,16.5,San Antonio Spurs,-105,-124,0.512,0.554,61.07,-68.39,21.9,22.0,5.78,-0.934,0.825,0.175,1.0,0.9,0.80,29.02,2.94,0.21,0.03,27.0,1.0,18.5,245.5,110.1,3,100.82,12
301,Quentin Grimes,NaN,PTS+REB,12.5,Miami Heat,-125,100,0.556,0.500,60.92,-78.80,23.9,24.0,9.13,-1.249,0.894,0.106,0.8,0.9,0.80,33.04,3.92,0.23,0.07,19.0,5.0,-2.0,242.5,112.8,9,104.50,1
213,Quentin Grimes,NaN,PTS+REB+AST,16.0,Miami Heat,-137,-137,0.578,0.578,52.58,-79.59,27.5,26.5,9.71,-1.184,0.882,0.118,0.6,0.8,0.67,33.04,3.92,0.23,0.07,22.0,5.0,-2.0,242.5,112.8,9,104.50,1
302,Daniel Gafford,NaN,PTS+REB,19.5,Minnesota Timberwolves,100,-110,0.500,0.524,52.20,-54.37,24.6,24.5,7.18,-0.710,0.761,0.239,1.0,0.7,0.47,24.35,3.32,0.19,0.03,15.4,5.0,8.0,235.5,112.2,8,101.42,10
391,Collin Sexton,NaN,PTS+AST,18.5,San Antonio Spurs,-114,-110,0.533,0.524,50.36,-62.01,24.2,24.5,6.75,-0.844,0.801,0.199,0.8,0.9,0.60,26.21,6.60,0.25,0.04,22.2,5.0,18.5,245.5,110.1,3,100.82,12
214,Daniel Gafford,NaN,PTS+REB+AST,20.5,Minnesota Timberwolves,-108,-105,0.519,0.512,50.03,-56.85,26.2,25.5,7.41,-0.769,0.779,0.221,1.0,0.7,0.47,24.35,3.32,0.19,0.03,16.2,5.0,8.0,235.5,112.2,8,101.42,10


In [11]:
# output_path = f'data/props/ev_analysis/underdog.csv'
output_path = f'data/props/ev_analysis/prizepicks.csv'
df.to_csv(output_path, index=False)